#### Topic classification and paragraph splitting with LLM 
- as of July 24, there is a bug with vllm 0.52 , have to build from source . follow: https://docs.vllm.ai/en/latest/getting_started/installation.html#build-from-source
- or just use vllm 0.51, but it won't work with gemma 2
- reference https://github.com/vllm-project/vllm/issues/6462
- also go to run gemma 2 , need to install ; pip install flashinfer -i https://flashinfer.ai/whl/cu121/torch2.3

In [1]:
import os,sys
sys.path.insert(0,'../libs')
sys.path.insert(0,'../src')
from utils import load_json
import openai
from llm_utils import BSAgent,custom_llm_result_parsing,donload_hf_model
import numpy as np
import pandas as pd
import re,json,copy
from tqdm import tqdm
from prompts import topic_identify_simple_pt, output_fixing_pt
import pprint

/data/home/xiong/miniconda3/envs/gpt/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from typing import Optional,List,Literal,Iterable
from langchain.output_parsers import PydanticOutputParser
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

#### Download model from model hub

In [3]:
# REPO_ID = "Qwen/Qwen2-7B-Instruct"
# REPO_ID = "Qwen/Qwen2-72B-Instruct"
# REPO_ID = "google/gemma-2-9b-it" 
# REPO_ID = "google/gemma-2-27b-it"
# REPO_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# REPO_ID = "meta-llama/Meta-Llama-3.1-70B-Instruct"
# save_location = os.path.join('/root/data/hf_cache',REPO_ID)
# donload_hf_model(REPO_ID,save_location)

#### user vllm to serve api endpoint

- #### serve opensource model as openai end point
- CUDA_VISIBLE_DEVICES=6,7  python -m vllm.entrypoints.openai.api_server --model /root/data/hf_cache/Qwen/Qwen2-7B-Instruct --dtype auto --api-key token-abc123 --port 8000
- CUDA_VISIBLE_DEVICES=4,5,6,7  python -m vllm.entrypoints.openai.api_server --model /root/data/hf_cache/Qwen/Qwen2-72B-Instruct --dtype auto --api-key token-abc123 --port 8000 --tensor-parallel-size 4
- CUDA_VISIBLE_DEVICES=4,5,6,7  python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-72B-Instruct --dtype auto --api-key token-abc123 --port 8000 --tensor-parallel-size 4
- CUDA_VISIBLE_DEVICES=6,7  python -m vllm.entrypoints.openai.api_server --model /root/data/hf_cache/meta-llama/Meta-Llama-3.1-8B-Instruct --dtype auto --api-key token-abc123 --port 8000
- CUDA_VISIBLE_DEVICES=4,5,6,7  python -m vllm.entrypoints.openai.api_server --model /root/data/hf_cache/meta-llama/Meta-Llama-3.1-70B-Instruct --dtype auto --api-key token-abc123 --port 8000 --tensor-parallel-size 4
- export VLLM_ATTENTION_BACKEND=FLASHINFER
- CUDA_VISIBLE_DEVICES=6,7  python -m vllm.entrypoints.openai.api_server --model /root/data/hf_cache/google/gemma-2-27b-it --dtype auto --api-key token-abc123 --port 8000 --tensor-parallel-size 2
- export VLLM_ATTENTION_BACKEND=FLASHINFER
- CUDA_VISIBLE_DEVICES=6,7  python -m vllm.entrypoints.openai.api_server --model /root/data/hf_cache/google/gemma-2-9b-it --dtype auto --api-key token-abc123 --port 8000 --tensor-parallel-size 2

In [4]:
from dotenv import load_dotenv, find_dotenv
env_path = '../.env'
load_dotenv(dotenv_path=env_path)

True

In [5]:
## Modify OpenAI's API key and API base to use vLLM's API server.
# openai_api_key = "token-abc123"
# openai_api_base = "http://localhost:8000/v1"

# llm_agent  = BSAgent(api_key=openai_api_key,
#                      api_base_url=openai_api_base,
#                     temperature=0)
# llm_agent.model = llm_agent.client.models.list().data[0].id

## test third party api
openai_api_key = os.getenv("Netmind_AIP_KEY")
openai_api_base = os.getenv("Netmind_BASEURL")

llm_agent  = BSAgent(api_key=openai_api_key,
                     api_base_url=openai_api_base,
                    temperature=0)
llm_agent.model = "meta-llama-Meta-Llama-3.1-70B-Instruct"

print('current running model :{}'.format(llm_agent.model))

current running model :meta-llama-Meta-Llama-3.1-70B-Instruct


#### Try one example

In [6]:
## just run one test, make sure the api works 
pt = {'System':'You are a helpful assistant.',
      'Human':'Who won the world series in 2020?'}
if 'gemma' in llm_agent.model:
    pt = {'Human':'Who won the world series in 2020?'}  
res = llm_agent.get_response_content(prompt_template=pt, 
                                     #max_tokens=4096,
                                     temperature=0)
print(res)  

The Los Angeles Dodgers won the 2020 World Series, defeating the Tampa Bay Rays in the series 4 games to 2.


#### Set response fromat

In [7]:
class Topic(BaseModel):
    topic_label: Literal["Economic Outlook", "Monetary Policy", "Fiscal Stance", 
                         "Financial Stability","External Stance","Climate Change",
                         "Inclusion","Technology","Governance","Structural Reforms",
                         "Other Topics"] = Field(description="short title for the topic")
    confidence_score: int = Field(description="confidence score of topic identification, ranges from 0 to 100")

class Topic_Result(BaseModel):
    reasoning: str = Field(description="the reasoning process for topic identification")
    topic_labels: List[Topic] = Field(description="list of topics that are identified")
    
parser = PydanticOutputParser(pydantic_object=Topic_Result)

#pprint.pprint(parser.get_format_instructions())
example_output = '{"reasoning": "test test test", "topic_labels": [{"topic_label":"Other Topics","confidence_score":10}]}'
parser.parse(example_output)

Topic_Result(reasoning='test test test', topic_labels=[Topic(topic_label='Other Topics', confidence_score=10)])

#### Try one paragraph and see how well it works 

In [8]:
df = pd.read_csv('/data/home/xiong/data/Fund/CSR/detailed_topic_identification_eval_small.csv')
df.head()

,paragraph,topic_model_label,mapped_topic_label,ground_truth_label,Notes
0,"Additional, targeted fiscal support is warrant...",Fiscal Management,Fiscal Stance,Fiscal Stance,NaN
1,The authorities agreed that additional fiscal ...,Fiscal Management,Fiscal Stance,Fiscal Stance,NaN
2,Box 1. Jordan: Past Fund Staff Advice and Impl...,Fiscal Management,Fiscal Stance,Fiscal Stance,NaN
3,The authorities consider that geopolitical ten...,Financial Stability,Financial Stability,Financial Stability,NaN
4,The banking sector is in good financial health...,Financial Stability,Financial Stability,Financial Stability,NaN


In [9]:
df.rename(columns={"paragraph": "examples"}, inplace=True)
print(df.iloc[2]['ground_truth_label'])
print(df.iloc[2]['examples'])
test_input = df.iloc[2]

Fiscal Stance
Box 1. Jordan: Past Fund Staff Advice and Implementation Macroeconomic policies were implemented with delays but broadly in line with past staff advice. Despite efforts, however, fiscal consolidation fell short of expectations, reflecting sizable revenue shortfalls— due to widespread weaknesses in tax administration, low-quality measures, delays in implementation, and some tax policy reversals—and public financial management deficiencies.
Improving the tax framework. Macro-critical measures, such as the reduction of sales-tax exemptions and amendments to the income tax law, have strengthened the tax policy framework. However, persistent weaknesses in tax administration, inability to tackle tax evasion and smuggling, and some tax policy reversals contributed to sizable revenue slippages over 2018-19. To reduce deviations from program targets, the authorities have often relied on low-quality measures, including cuts to capital spending and postponing the clearance of domest

#### Run one basic economic sector question and see if LLM knows basic macro

In [10]:
if 'gemma' in llm_agent.model:
    topic_identify_simple_pt['Human']  = topic_identify_simple_pt.get('System') + topic_identify_simple_pt.get('Human') 
    del topic_identify_simple_pt['System']

In [11]:
#print(topic_identify_simple_pt['Human'].format(PARA=test_input['examples']))
topic_pt_temp = copy.copy(topic_identify_simple_pt)
topic_pt_temp['Human']=topic_identify_simple_pt['Human'].format(PARA=test_input['examples'])
res = llm_agent.get_response_content(prompt_template=topic_pt_temp,max_tokens=4096)

In [12]:
## two different ways of parsing results 
res_dict = llm_agent.parse_load_json_str(res) 
print(res_dict)
res_dict2 = parser.parse(res)
print(res_dict2)

{'reasoning': "The paragraph discusses Jordan's implementation of macroeconomic policies, including fiscal consolidation, tax administration, and structural reforms. It also touches on the country's efforts to improve the business environment, promote inclusive growth, and address labor market participation. The paragraph provides an assessment of the country's progress in implementing these policies and reforms, highlighting both successes and challenges. Based on the content, the paragraph appears to be discussing the country's economic policies and reforms, which aligns with the topic of Structural Reforms. However, it also extensively discusses fiscal consolidation, tax administration, and monetary policy, which are key aspects of Fiscal Stance and Monetary Policy. Therefore, I am assigning multiple topic labels to the paragraph.", 'topic_labels': [{'topic_label': 'Structural Reforms', 'confidence_score': 80}, {'topic_label': 'Fiscal Stance', 'confidence_score': 70}, {'topic_label'

In [13]:
sample_results = []
for idx,r in tqdm(df.iterrows()):
    topic_pt_temp = copy.copy(topic_identify_simple_pt)
    topic_pt_temp['Human']=topic_identify_simple_pt['Human'].format(PARA=r['examples'])
    res = llm_agent.get_response_content(prompt_template=topic_pt_temp,max_tokens=4096)
    try:
        res_dict = parser.parse(res).dict()
        sample_results.append([r['ground_truth_label'],r['examples'],res,res_dict['reasoning'],res_dict['topic_labels']])
    except:
        sample_results.append([r['ground_truth_label'],r['examples'],res,None,None])

100it [06:11,  3.72s/it]


In [14]:
res_df = pd.DataFrame(sample_results,columns=['ground_truth_label','paragraph','llm_raw_response','reasoning','topic_labels'])
res_df.head(10)

,ground_truth_label,paragraph,llm_raw_response,reasoning,topic_labels
0,Fiscal Stance,"Additional, targeted fiscal support is warrant...","```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the need for...,"[{'topic_label': 'Fiscal Stance', 'confidence_..."
1,Fiscal Stance,The authorities agreed that additional fiscal ...,"```json\n{""reasoning"": ""The paragraph primaril...",The paragraph primarily discusses the French a...,"[{'topic_label': 'Fiscal Stance', 'confidence_..."
2,Fiscal Stance,Box 1. Jordan: Past Fund Staff Advice and Impl...,"```json\n{""reasoning"": ""The paragraph discusse...",The paragraph discusses Jordan's implementatio...,"[{'topic_label': 'Structural Reforms', 'confid..."
3,Financial Stability,The authorities consider that geopolitical ten...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the stabilit...,"[{'topic_label': 'Financial Stability', 'confi..."
4,Financial Stability,The banking sector is in good financial health...,"```json\n{""reasoning"": ""The paragraph primaril...",The paragraph primarily discusses the financia...,"[{'topic_label': 'Financial Stability', 'confi..."
5,Financial Stability,The macroprudential stance is broadly appropri...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the macropru...,"[{'topic_label': 'Financial Stability', 'confi..."
6,Monetary Policy,Durably reducing inflation to its target and a...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the need for...,"[{'topic_label': 'Monetary Policy', 'confidenc..."
7,Monetary Policy,The authorities noted that the current monetar...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the current ...,"[{'topic_label': 'Monetary Policy', 'confidenc..."
8,Monetary Policy,The BSP’s prompt action to fight inflation is ...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the BSP's (B...,"[{'topic_label': 'Monetary Policy', 'confidenc..."
9,Fiscal Stance,The authorities broadly agreed with the need f...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily focuses on the authori...,"[{'topic_label': 'Fiscal Stance', 'confidence_..."


##### Check for output parsing errors 

In [15]:
if len(res_df[res_df['reasoning'].isna()])>0:
    error_example = res_df[res_df['reasoning'].isna()].iloc[0]
    #print(error_example['llm_raw_response'])
    output_fixing_pt_temp = copy.copy(output_fixing_pt)
    output_fixing_pt_temp['Human']=output_fixing_pt_temp['Human'].format(LLM_OUTPUT=error_example['llm_raw_response'])
    r_dict = custom_llm_result_parsing(error_example['llm_raw_response'],llm_agent,json_parse=True,parser=parser,output_fixing_pt_temp=output_fixing_pt_temp,verbose=False)
    print(r_dict)

In [16]:
error_fixing_labels = []
for idx,r in tqdm(res_df.iterrows()):
    if pd.isna(r['reasoning']):
        output_fixing_pt_temp = copy.copy(output_fixing_pt)
        output_fixing_pt_temp['Human']=output_fixing_pt_temp['Human'].format(LLM_OUTPUT=r['llm_raw_response'])
        res_dict = custom_llm_result_parsing(r['llm_raw_response'], llm_agent,json_parse=True, parser=parser, output_fixing_pt_temp=output_fixing_pt_temp)
        #print(res_dict)
        if res_dict:
            error_fixing_labels.append(res_dict['topic_labels'])
        else:
            error_fixing_labels.append([])
    else:
        error_fixing_labels.append(r['topic_labels'])

100it [00:00, 20055.97it/s]


In [17]:
res_df['topic_labels_processed_1'] = error_fixing_labels
res_df['predicted_topic_labels'] = [[d['topic_label'] for d in sublist] for sublist in res_df['topic_labels_processed_1']]
res_df['is_in_predicted'] = res_df.apply(lambda row: row['ground_truth_label'] in row['predicted_topic_labels'], axis=1)
res_df.head(10)

,ground_truth_label,paragraph,llm_raw_response,reasoning,topic_labels,topic_labels_processed_1,predicted_topic_labels,is_in_predicted
0,Fiscal Stance,"Additional, targeted fiscal support is warrant...","```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the need for...,"[{'topic_label': 'Fiscal Stance', 'confidence_...","[{'topic_label': 'Fiscal Stance', 'confidence_...",[Fiscal Stance],True
1,Fiscal Stance,The authorities agreed that additional fiscal ...,"```json\n{""reasoning"": ""The paragraph primaril...",The paragraph primarily discusses the French a...,"[{'topic_label': 'Fiscal Stance', 'confidence_...","[{'topic_label': 'Fiscal Stance', 'confidence_...",[Fiscal Stance],True
2,Fiscal Stance,Box 1. Jordan: Past Fund Staff Advice and Impl...,"```json\n{""reasoning"": ""The paragraph discusse...",The paragraph discusses Jordan's implementatio...,"[{'topic_label': 'Structural Reforms', 'confid...","[{'topic_label': 'Structural Reforms', 'confid...","[Structural Reforms, Fiscal Stance, Monetary P...",True
3,Financial Stability,The authorities consider that geopolitical ten...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the stabilit...,"[{'topic_label': 'Financial Stability', 'confi...","[{'topic_label': 'Financial Stability', 'confi...",[Financial Stability],True
4,Financial Stability,The banking sector is in good financial health...,"```json\n{""reasoning"": ""The paragraph primaril...",The paragraph primarily discusses the financia...,"[{'topic_label': 'Financial Stability', 'confi...","[{'topic_label': 'Financial Stability', 'confi...",[Financial Stability],True
5,Financial Stability,The macroprudential stance is broadly appropri...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the macropru...,"[{'topic_label': 'Financial Stability', 'confi...","[{'topic_label': 'Financial Stability', 'confi...",[Financial Stability],True
6,Monetary Policy,Durably reducing inflation to its target and a...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the need for...,"[{'topic_label': 'Monetary Policy', 'confidenc...","[{'topic_label': 'Monetary Policy', 'confidenc...",[Monetary Policy],True
7,Monetary Policy,The authorities noted that the current monetar...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the current ...,"[{'topic_label': 'Monetary Policy', 'confidenc...","[{'topic_label': 'Monetary Policy', 'confidenc...",[Monetary Policy],True
8,Monetary Policy,The BSP’s prompt action to fight inflation is ...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily discusses the BSP's (B...,"[{'topic_label': 'Monetary Policy', 'confidenc...","[{'topic_label': 'Monetary Policy', 'confidenc...",[Monetary Policy],True
9,Fiscal Stance,The authorities broadly agreed with the need f...,"```json\n{\n ""reasoning"": ""The paragraph prim...",The paragraph primarily focuses on the authori...,"[{'topic_label': 'Fiscal Stance', 'confidence_...","[{'topic_label': 'Fiscal Stance', 'confidence_...",[Fiscal Stance],True


In [18]:
# Calculate the accuracy rate
accuracy_rate = res_df['is_in_predicted'].mean()
print(f'Accuracy Rate: {accuracy_rate * 100:.2f}%')

Accuracy Rate: 92.00%


In [19]:
#res_df.to_csv('/root/data/Fund/CSR/llm_detailed_topic_results_Qwen2-72b.csv')